# The AgMIP calibration protocol with diffWOFOST

![diffWOFOST logo](https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png)

The AgMIP calibration protocol provides a structured approach to crop-model calibration using multiple observation types. It specifies how to group observations, select parameters, estimate their values, and evaluate model fit.

This notebook applies the protocol to the public French winter-wheat dataset from AgMIP Calibration Phase IV, using **diffWOFOST**, a differentiable implementation of WOFOST. In a differentiable model the simulated observations are a smooth function of the parameters, so the sum of squares has a gradient and Adam can follow it. That gradient search is compared with the reference implementation's multi-start Nelder–Mead search.

The workflow follows the protocol's eight steps: documenting the model and data, calibrating phenology, calibrating biomass and nitrogen-related variables, selecting parameters, jointly refitting the selected parameters, and evaluating goodness of fit. The objective functions and selection procedure are intended to follow the protocol; model-specific adaptations are documented below.


## Dataset

AgMIP Calibration Phase IV uses the French winter-wheat cultivar Apache from Arvalis variety trials. The dataset covers 22 site-years: 14 calibration environments and eight held-out evaluation environments, separated by site and year. The public data are available in [https://github.com/sbuis/AgMIP-Calibration-Phase-IV](https://github.com/sbuis/AgMIP-Calibration-Phase-IV).

The input file covers all 22 environments, but the public observation file contains only the 14 calibration environments. The official evaluation observations are not available, so this notebook can assess calibration fit but cannot independently evaluate predictive performance on the official evaluation set. See Wallach et al. (2024, 2025) for the AgMIP protocol and evaluation context.

The observation file does not contain the `Date_BBCH10` variable listed in the units table. Emergence dates are therefore unavailable for calibration. A separate [synthetic dataset](https://github.com/sbuis/AgMIP_calibration_PhaseIV_step2_synthetic_experiment) represents the same environments simulated with STICS and known generating parameters; it is useful for testing parameter recovery, but it is distinct from the real-data experiment considered here.


## Applying the protocol to WOFOST

The AgMIP protocol is model-independent, but its application depends on the observations and simulated variables available in WOFOST. The main adaptations are:

- **Phenology:** heading (BBCH55) is used as a proxy for anthesis (DVS 1), and maturity (BBCH90) represents DVS 2. BBCH30 is not fitted because it has no distinct WOFOST development-stage threshold. Emergence dates (BBCH10) are unavailable, so `TSUMEM` remains fixed at its Arminda default.
- **Observation groups:** biomass is fitted on a logarithmic scale; yield uses `TWSO`; plant nitrogen is harvest nitrogen as a percentage of above-ground biomass; and grain protein is estimated as 5.7 times the nitrogen percentage of `TWSO`. Grain and ear number are not fitted because corresponding WOFOST states are unavailable.
- **Parameters and selection:** calibration starts from Arminda cultivar parameters. Major parameters are fitted first, then candidate parameters are tested sequentially and retained according to AICc.
- **Optimization:** Adam is compared with multi-start Nelder–Mead. The optimizer is not prescribed by the protocol, but the objective functions, selection procedure, and final weighted refit are preserved.
- **Evaluation:** the 14 public site-years are used for calibration. Without observations from the eight held-out environments, independent predictive evaluation is not possible here.

Soil hydraulic properties, rooting depth, initial mineral nitrogen, organic-matter properties, and fertilizer transformations are fixed inputs rather than calibrated parameters. The following sections document the principal assumptions and model-specific choices.


## Setup

### Optimization settings

Adam uses 20 Latin-hypercube starts for groups with two major parameters and five for groups with one major parameter or for each candidate fit. For candidate fits, one start uses the best parameter vector from the preceding fit; the other starts explore the permitted bounds. The weighted refit uses 20 starts, following the default in `CroptimizR::run_protocol_agmip`.

The initial learning rate is 0.25 for phenology and 0.12 for the later groups. After three steps without a new best objective, the optimizer returns to the best vector, halves the learning rate, and resets Adam's momentum, up to four times. At the minimum learning rate, it stops after ten steps without improvement or at 80 steps, whichever comes first. These settings do not guarantee convergence; interpret results alongside the stopping diagnostics.

### Saved results and reproducibility

The tables and figures below are stored with this document. The search is run by `agmip_retrain.py`, which loads the model from `agmip_calibration.py` and writes the JSON files next to the notebook.

## Steps 1–5: applying the AgMIP protocol to WOFOST

**Step 1 — Default parameters.** The starting point is the Arminda cultivar parameterization from the WOFOST 8.1 winter-wheat parameter file. Apache, the target cultivar, is a French soft winter wheat. Arminda is used as the reference cultivar; parameters not selected for estimation remain at their Arminda defaults. Arminda uses vernalization (`IDSL = 2`). The WOFOST 8.1 nitrogen parameters were estimated for Arminda (Berghuijs et al., 2024).

**Steps 2–3 — Observations and variable groups.** Observations are grouped according to the WOFOST outputs they constrain. Variables without suitable simulated counterparts, or without the required measurements, are excluded.

**Step 4 — Major parameters.** Each group is first calibrated using a small set of parameters with clear mechanistic links to the observations. Heading (BBCH55) is used as a proxy for anthesis because the observations do not record anthesis directly.

**Step 5 — Candidate parameters.** Additional parameters are tested sequentially and retained only if they improve AICc. For phenology, vernalization parameters are tested before daylength because the crops are autumn-sown and experience different winter temperatures; the trial sites' relatively narrow latitude range (approximately 47–50°N) limits, but does not eliminate, photoperiod variation.

| Variable group | Observations used | Major parameters | Rationale |
|---|---|---|---|
| Phenology | Heading (BBCH55) and maturity (BBCH90) dates | `TSUM1`, `TSUM2` | Development to heading/anthesis proxy and maturity. |
| Biomass | Log-transformed above-ground biomass, `ln(TAGP)` | `AMAX_REF`, `SPAN` | Reference assimilation capacity and leaf lifespan affect biomass accumulation. |
| Plant nitrogen | Harvest nitrogen as a percentage of above-ground biomass | `RNUPTAKEMAX` | Maximum nitrogen uptake rate. |
| Yield | Grain yield represented by `TWSO` | `REALLOC_STEM_FRACTION` | Controls the fraction of stem biomass available for reallocation to grain. |
| Grain protein | Protein estimated as 5.7 times the nitrogen percentage of `TWSO` | `NMAXSO` | Maximum nitrogen concentration in storage organs. |

For grain protein, `NRESIDLV`, `NRESIDST`, and `TCNT` are additional candidates. They affect nitrogen retention in vegetative tissues and transfer to grain. The observation file lacks emergence dates, and grain and ear number are not fitted because corresponding WOFOST states are unavailable.


### Observed-to-simulated variable mapping

| Observed | Simulated | Group | Order | Note |
| --- | --- | --- | --- | --- |
| Date_BBCH30 | none | phenology |  | Stem elongation is not a WOFOST stage. |
| Date_BBCH55 | day when DVS reaches 1 | phenology | 1 | Heading, used as anthesis. |
| Date_BBCH90 | day when DVS reaches 2 | phenology | 1 | Maturity. |
| Biomass | TAGP | plant_biomass | 2 | g m-2 in the file, kg ha-1 in WOFOST. |
| N_in_biomassHarvest | crop N / TAGP | plant_N_content | 3 | Percent. Needs SNOMIN. |
| Grain_Yield | TWSO | yield | 4 | g m-2 in the file, kg ha-1 in WOFOST. |
| ProteinContentGrain | storage-organ N / TWSO | seed_protein | 5 | Percent. |
| EarsPerSqM | none |  |  | WOFOST does not simulate ear number. |
| Grain_Number | none |  |  | WOFOST does not simulate grain number. |

### Parameter selection

| Group | Role | Parameter | Why this one |
| --- | --- | --- | --- |
| phenology | major | TSUM1 | Temperature sum, emergence to anthesis. Moves BBCH55. |
| phenology | major | TSUM2 | Temperature sum, anthesis to maturity. Moves BBCH90. |
| phenology | candidate | VERNSAT | Saturated vernalization requirement. |
| phenology | candidate | VERNBASE | Base vernalization requirement. |
| phenology | candidate | DLO | Optimum daylength. |
| plant_biomass | major | AMAX_REF | Assimilation level. One of the two allowed majors. |
| plant_biomass | major | SPAN | Leaf lifespan. Sets how long biomass keeps accumulating. |
| plant_biomass | candidate | TDWI | Initial biomass. Early-season residual. |
| plant_biomass | candidate | RGRLAI | Early canopy closure. Tried on biomass only; there is no LAI to fit. |
| plant_N_content | major | RNUPTAKEMAX | Maximum daily nitrogen uptake. |
| plant_N_content | candidate | NMAXSO | Maximum nitrogen concentration in the grain. |
| yield | major | REALLOC_STEM_FRACTION | Stem biomass that can move into the grain. |
| yield | candidate | CVO | Conversion efficiency into storage organs. |
| seed_protein | major | NMAXSO | Same parameter as above, if it was not already selected. |
| seed_protein | candidate | NRESIDLV | Residual nitrogen in leaves. Lowering it releases more nitrogen to the grain. |
| seed_protein | candidate | NRESIDST | Residual nitrogen in stems. |
| seed_protein | candidate | TCNT | Days over which translocatable nitrogen moves into the grain. |

## Modeling choices and assumptions

The protocol specifies the calibration sequence, but its application also requires choices about error metrics, fertilizer inputs, soil properties, and parameter identifiability. These choices are fixed inputs or modeling assumptions, not additional calibrated parameters.

### Error metric and observation scales

Each observation group has its own sum-of-squares objective. Biomass is fitted on a logarithmic scale because absolute squared errors can otherwise be dominated by the largest measurements. For example, a 20% error at 2,000 g m⁻² contributes far more to raw-scale squared error than the same relative error at 100 g m⁻². The biomass residual is `log(TAGP_obs) - log(TAGP_sim)`, which approximates relative error for small differences.

The other groups use their specified scales: phenology in days, yield in g m⁻², plant nitrogen as a percentage of above-ground biomass, and grain protein as a percentage. Yield, plant nitrogen, and grain protein provide one harvest observation per environment, unlike the repeated biomass measurements.

### Fertilizer representation in SNOMIN

The trial file records application date, product, and nitrogen rate (kg N ha⁻¹). SNOMIN requires the application amount, depth, organic-matter fraction, C:N ratio, initial age, and ammonium/nitrate fractions. The products are represented as mineral fertilizers, so organic-matter fraction is zero and the organic-matter C:N and age settings do not affect these applications.

- **Ammonitrate:** half the nitrogen is assigned to ammonium and half to nitrate.
- **18_46:** all nitrogen is assigned to ammonium.
- **Solution azotée:** 75% is assigned to ammonium and 25% to nitrate. The urea share is treated as ammonium because this configuration does not explicitly represent urea hydrolysis. This omits the short hydrolysis delay and does not explicitly represent ammonia volatilization.

The application depth is assumed to be 10 cm because it is not specified in the input file. Since amendments are represented within the soil-layer structure, effective placement depends on the upper-layer thickness (25–30 cm in these profiles).

### Soil hydraulic properties and nitrogen assumptions

The soil file describes horizon thickness, texture, bulk density, field capacity, wilting point, stones, pH, and organic matter. The Wösten pedotransfer function estimates van Genuchten-type water-retention parameters and hydraulic conductivity from texture, organic matter, bulk density, and topsoil status. Measured field capacity and wilting point are used to validate the conversion.

SNOMIN requires an initial soil-organic-matter C:N ratio, which is not supplied in the input file. The implementation fixes `CNRatioSOMI` at approximately 10 and the apparent age of the initial organic matter at 24 years, following the SoilGrids example. Soil hydraulic properties, rooting depth, initial mineral nitrogen, and organic-matter properties are therefore prescribed, not calibrated. Their uncertainty may affect the crop-parameter estimates.

### Phenology and parameter identifiability

Phenology is calibrated using `DVS_Phenology`, which calculates development from temperature and cultivar parameters without the soil water and nitrogen modules. The subsequent biomass, nitrogen, yield, and protein fits use `Wofost81` with `SoilModuleWrapper_NWLP_MLWB_SNOMIN`. The joint refit in Step 7 stays with that full model. Each site-year is simulated once, and the heading and maturity residuals are read from its development stage together with biomass, nitrogen, yield, and protein.

`TSUMEM` is the temperature sum from sowing to emergence; `TSUM1` controls development from emergence to anthesis, and `TSUM2` controls development from anthesis to maturity. Because emergence observations are missing, `TSUMEM` and `TSUM1` cannot be identified independently from heading and maturity dates alone: their effects on the total thermal-time requirement up to heading can compensate. `TSUMEM` is therefore fixed at the Arminda default of 120 °C day, while `TSUM1` and `TSUM2` are estimated.


## Step 6: calibrating and selecting phenology parameters

### Major parameters

Phenology is fitted first using heading (BBCH55) and maturity (BBCH90) dates from 14 calibration environments, giving 28 observations. `TSUM1` and `TSUM2` are estimated from 20 Latin-hypercube initializations within the allowed bounds. The best fit found is used for subsequent candidate selection; parameters not selected for estimation remain at their Arminda defaults. Heading is used as a proxy for anthesis.

### Candidate selection using AICc

AICc balances goodness of fit against the number of estimated parameters, with a stronger complexity penalty when the sample is small. Under the independent Gaussian residual assumption, the criterion used here is

$$
\mathrm{AICc} = n\ln(\mathrm{SS}/n)+2p+\frac{2p(p+1)}{n-p-1},
$$

where $n$ is the number of observations, $SS$ is the minimized sum of squares, and $p$ is the number of estimated parameters. Lower AICc is preferred. A candidate is retained only if adding it reduces AICc relative to the best model considered so far.

The saved search tables below report the fitted results and candidate decisions. Candidate parameters not retained remain at their Arminda defaults. The optimizer comparison is incomplete for Adam's later grain-protein candidates; the phenology search itself is available for both methods.

### Residual variance for the joint refit

For each observation group, the residual variance is estimated as $SS/(n-p)$, where $p$ is the number of retained parameters. This variance estimate is distinct from the $SS/n$ term in the AICc expression. Step 7 scales each group's sum of squares by its estimated variance, so groups with larger residual variance contribute less weight. This scaling does not change the optimum of a group fitted alone, but parameters can change in the joint refit when all groups contribute to the objective.

### Smooth dates for optimization

A hard stage-crossing date is not necessarily differentiable with respect to model parameters. Gradient-based fitting therefore uses a smooth approximation to the stage-crossing date, while reported phenology errors are calculated from actual simulated crossing dates. This allows gradients to guide optimization while keeping the evaluation metric in calendar days.


### Model selection results

| Search | Parameters retained | Sum of squares | AICc |
| --- | --- | --- | --- |
| Adam | TSUM1, TSUM2 | 839.1 | 99.68 |
| Nelder–Mead | TSUM1, TSUM2 | 837.9 | 99.64 |

### Estimated parameter value

| Parameter | Arminda default | Adam | Nelder–Mead |
| --- | --- | --- | --- |
| TSUM1 (°C day) | 886.0 | 618.4 | 618.4 |
| TSUM2 (°C day) | 860.0 | 956.1 | 961.4 |

Heading and maturity errors after the phenology fit, in days. Bias is simulated minus observed.

### Phenology fit performance

| Stage | Parameters | n | bias | RMSE |
| --- | --- | ---: | ---: | ---: |
| BBCH55 | Arminda default | 14 | 18.86 | 19.10 |
| BBCH55 | after step 6 | 14 | -0.93 | 2.55 |
| BBCH90 | Arminda default | 14 | 11.57 | 13.67 |
| BBCH90 | after step 6 | 14 | 2.14 | 7.46 |

### Estimated phenology parameters

| Parameter | Role | Default | Estimated |
| --- | --- | ---: | ---: |
| TSUM1 | selected | 886.0 | 618.43 |
| TSUM2 | selected | 860.0 | 956.09 |
| VERNSAT | default | 44.0 | 44.00 |
| VERNBASE | default | 9.0 | 9.00 |
| DLO | default | 16.3 | 16.30 |

## Step 6: calibrating biomass, nitrogen, yield and grain protein

With the selected phenology parameters fixed, the remaining four variable groups are calibrated separately. Each group has its own observations, major parameters, and candidate parameters; candidates are retained only when they reduce AICc.

### Observation scales and conversions

- **Biomass:** observed `TAGP` is in g m⁻², while WOFOST reports kg ha⁻¹. Observations are multiplied by 10 to convert units, then fitted on a logarithmic scale so that relative errors are not dominated by large measurements.
- **Yield:** observed grain yield is in g m⁻²; simulated `TWSO` is divided by 10 to convert kg ha⁻¹ to g m⁻².
- **Plant nitrogen:** harvest nitrogen is expressed as a percentage of above-ground biomass, including leaves, stems, and grain.
- **Grain protein:** protein is estimated as 5.7 times the nitrogen concentration of `TWSO`. Zero protein values in the observation file are treated as missing, not as measured zeros.

Soil hydraulic properties, rooting depth, and soil-nitrogen inputs remain fixed as described above.

### Optimization and variance estimates

Each group is optimized with Adam, starting at a learning rate of 0.12. When progress stalls, the optimizer returns to its best solution and halves the learning rate; the stopping rules are described in Setup. The best objective found is used to calculate AICc. The residual variance for each group is estimated as $SS/(n-p)$ and carried into Step 7, where the selected parameters from all groups are jointly fitted using a weighted objective.


## Step 8: evaluating the calibration environments

The 14 calibration environments are used both to estimate parameters and to assess model fit. This is an **in-sample assessment**, not an independent test of predictive performance.

The table compares three stages: Arminda defaults, separate group fits from Step 6, and the joint weighted refit from Step 7. It reports bias (simulated minus observed), RMSE, and RRMSE (RMSE divided by the mean observed value). Biomass errors are reported on both the original scale (g m⁻²) and the logarithmic scale used in calibration. The figure compares simulated and observed values for the same environments and distinguishes the three parameter sets by marker shape.

**Default parameterization**

| Variable | n | Bias | RMSE | RRMSE |
| --- | ---: | ---: | ---: | ---: |
| BBCH55 | 14 | 18.857 | 19.105 | 0.092 |
| BBCH90 | 10 | 8.500 | 10.455 | 0.039 |
| Biomass | 44 | 198.744 | 242.828 | 0.270 |
| ln(Biomass) | 44 | 0.287 | 0.363 | 0.056 |
| Plant N | 9 | -0.228 | 0.312 | 0.252 |
| Yield | 13 | -136.242 | 294.504 | 0.341 |
| Protein | 13 | 0.349 | 0.951 | 0.086 |

**After step 6**

| Variable | n | Bias | RMSE | RRMSE |
| --- | ---: | ---: | ---: | ---: |
| BBCH55 | 14 | -0.929 | 2.550 | 0.012 |
| BBCH90 | 14 | 2.143 | 7.464 | 0.028 |
| Biomass | 44 | -32.227 | 114.151 | 0.127 |
| ln(Biomass) | 44 | -0.015 | 0.173 | 0.027 |
| Plant N | 9 | -0.011 | 0.200 | 0.161 |
| Yield | 13 | 21.937 | 170.427 | 0.198 |
| Protein | 13 | 0.008 | 0.888 | 0.080 |

**After step 7**
| Variable | n | Bias | RMSE | RRMSE |
| --- | ---: | ---: | ---: | ---: |
| BBCH55 | 14 | -0.357 | 2.464 | 0.012 |
| BBCH90 | 14 | 2.571 | 7.792 | 0.030 |
| Biomass | 44 | -28.046 | 103.595 | 0.115 |
| ln(Biomass) | 44 | -0.006 | 0.172 | 0.026 |
| Plant N | 9 | -0.003 | 0.184 | 0.149 |
| Yield | 13 | 9.712 | 133.393 | 0.155 |
| Protein | 13 | 0.003 | 0.893 | 0.080 |

![Calibration site-years.](figures/calibration_pairs.png)

*Calibration site-years.*

## How the fit changes through the protocol

Each line is one calibration environment. The simulated value is shown for the Arminda default, after the separate group fits (step 6), and after the joint refit (step 7). Biomass is the last measurement in each environment. A line that is flat after step 6 means the joint refit left that environment's simulation nearly unchanged. Four environments do not reach maturity with the Arminda default, so those BBCH90 lines start at step 6.

![Simulated values at the Arminda default, after step 6, and after step 7.](figures/protocol_stages.png)

*Simulated values at the Arminda default, after step 6, and after step 7.*

Relative RMSE at the same three stages. BBCH90 with the Arminda default uses the 10 environments that reach maturity.

| Stage | BBCH55 | BBCH90 | biomass | ln(biomass) | plant N | yield | protein |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| default | 0.092 | 0.039 | 0.270 | 0.056 | 0.252 | 0.341 | 0.086 |
| step 6 | 0.012 | 0.028 | 0.127 | 0.027 | 0.161 | 0.198 | 0.080 |
| step 7 | 0.012 | 0.030 | 0.115 | 0.026 | 0.149 | 0.155 | 0.080 |

Squared bias as a fraction of the mean squared error. A large fraction means the error is mostly a shift that is common to the environments.

| Variable | default | step 7 |
| --- | ---: | ---: |
| BBCH55 | 0.97 | 0.02 |
| BBCH90 | 0.66 | 0.11 |
| biomass | 0.67 | 0.07 |
| ln(biomass) | 0.62 | 0.00 |
| plant N | 0.53 | 0.00 |
| yield | 0.21 | 0.01 |
| protein | 0.13 | 0.00 |

## In-season biomass, crop nitrogen and soil mineral nitrogen

Above-ground biomass was measured repeatedly between March and harvest in 10 of the 14 calibration environments; plant nitrogen was measured only at harvest. Grey vertical lines mark mineral-fertilizer applications. The input file contains no manure applications, and these environments were not irrigated.

WOFOST's above-ground biomass (`TAGP`) includes living and dead leaves, stems, and grain. When a leaf class reaches the end of its lifespan, its biomass moves from the living-leaf pool to the dead-leaf pool; this transfer does not itself reduce total above-ground biomass. In this model configuration, nitrogen from senescing leaves is removed from the living-leaf pool at that day's concentration rather than retained in dead leaves or returned to the soil mineral-nitrogen pool. Nitrogen transferred to grain remains part of total above-ground nitrogen.

The soil curve shows ammonium plus nitrate across the soil profile. Fertilizer applications affect this pool, but the size and timing of the response depend on fertilizer composition, its representation in the soil layers, and subsequent soil processes; the change need not equal the applied nitrogen rate.

![Aboveground biomass. Grey lines are fertilizer dressings.](figures/trajectories_biomass.png)

*Aboveground biomass. Grey lines are fertilizer dressings.*

![Aboveground nitrogen. The circle is the harvest measurement.](figures/trajectories_nitrogen.png)

*Aboveground nitrogen. The circle is the harvest measurement.*

![Mineral nitrogen in the profile. Grey lines are fertilizer dressings.](figures/trajectories_soil.png)

*Mineral nitrogen in the profile. Grey lines are fertilizer dressings.*

## Comparing the calibration optimizers

Adam and Nelder–Mead are compared on three questions: whether they select similar parameters, whether their parameter estimates agree, and whether repeated starts satisfy the optimizer's stopping criterion. The tables summarize the saved results. SS and AICc are calculated separately for each variable group, with biomass fitted on the logarithmic scale.

Table A is Step 6. Each sum of squares uses that group's own objective, so a phenology value near 839 is not on the same scale as a biomass value near 1.3, and neither is the Step 7 score. Step 7 refits the selected parameters jointly on one full-model run per site-year. Its score divides each group's sum of squares by the variance from Adam's step 6 and adds the five terms. Heading and maturity are taken from that run. The Step 7 table below is the earlier search, which scored the dates with `DVS_Phenology` and then ran the full model again. A new search with the same 20 starts is using the single run.

Agreement on parameter selection does not imply agreement on parameter values or predictions.

The phenology diagnostics show the final objective values and parameter values from the 20 Nelder–Mead starts. Meeting the stopping criterion is distinct from finding a low objective value; a good fit alone does not establish convergence or parameter identifiability.


### A. Parameter selection

| Variable group | Adam selected parameters | Adam SS | Nelder–Mead selected parameters | Nelder–Mead SS |
| --- | --- | --- | --- | --- |
| Phenology | TSUM1, TSUM2 | 839 | TSUM1, TSUM2 | 838 |
| Biomass | AMAX_REF, SPAN, TDWI | 1.34 | AMAX_REF, SPAN, TDWI | 1.27 |
| Plant nitrogen | RNUPTAKEMAX | 0.398 | RNUPTAKEMAX | 0.279 |
| Yield | REALLOC_STEM_FRACTION | 3.56e+05 | REALLOC_STEM_FRACTION | 2.03e+05 |
| Grain protein | NMAXSO | 10.3 | NMAXSO | 10.4 |

**Search status.** The saved Nelder–Mead Step 6 search completed. The Adam search was interrupted during the grain-protein candidate tests: `NRESIDST` was cut off at step 37 and `TCNT` was not started. Therefore, those Adam candidate decisions are incomplete.

### B. Parameter estimates

| Parameter | Arminda default | Adam Step 6 | Adam Step 7 | Nelder–Mead Step 6 | Nelder–Mead Step 7 |
| --- | --- | --- | --- | --- | --- |
| TSUM1 | 886.00 | 618.428 | 629.387 | 618.447 | 618.387 |
| TSUM2 | 860.00 | 956.094 | 957.408 | 961.402 | 962.688 |
| AMAX_REF | 40.20 | 29.907 | 31.992 | 31.982 | 30.384 |
| SPAN | 35.00 | 51.785 | 35.742 | 34.101 | 50.023 |
| TDWI | 593.00 | 325.131 | 286.425 | 283.721 | 307.414 |
| RNUPTAKEMAX | 4.26 | 1.294 | 1.568 | 1.309 | 1.254 |
| REALLOC_STEM_FRACTION | 0.20 | 0.131 | 0.203 | 0.327 | 0.131 |
| NMAXSO | 0.02 | 0.020 | 0.019 | 0.019 | 0.020 |

### Step 7. Joint weighted score

This is not the sum of squares in table A. Each group's sum of squares is divided by the variance from Adam's step 6, and the five terms are added. A lower score is a better joint fit. Both searches finished all 20 starts on the earlier objective, which scored heading and maturity with `DVS_Phenology`. One Nelder–Mead start met its stopping test; the other 19 stopped at the evaluation ceiling.

| Search | Starts finished | Best weighted SS | Median weighted SS |
| --- | --- | --- | --- |
| Adam | 20 / 20 | 93.12 | 112.18 |
| Nelder–Mead | 20 / 20 | 111.81 | 266.25 |

![](figures/step7_starts.png)

### C. Phenology: sensitivity to initialization

![](figures/phenology_starts.png)

All 20 phenology starts met the stopping criterion. The finished sums of squares still differ, so meeting that criterion does not mean every start found the same solution.

### D. Candidate-selection details

| Group | Parameters tested | Adam SS | Adam AICc | Adam ΔAICc | Adam | Nelder–Mead SS | Nelder–Mead AICc | Nelder–Mead ΔAICc | Nelder–Mead |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Biomass | AMAX_REF, SPAN | 1.64 | -140.49 | — | Major | 1.64 | -140.52 | — | Major |
| Biomass | AMAX_REF, SPAN, TDWI | 1.34 | -146.92 | -6.43 | Retained | 1.27 | -149.24 | -8.72 | Retained |
| Biomass | AMAX_REF, SPAN, TDWI, RGRLAI | 1.34 | -144.59 | +2.34 | Rejected | 1.27 | -146.81 | +2.43 | Rejected |
| Grain protein | NMAXSO | 10.3 | -0.71 | — | Major | 10.4 | -0.51 | — | Major |
| Grain protein | NMAXSO, NRESIDLV | 9.84 | 1.58 | +2.30 | Rejected | 9.73 | 1.43 | +1.94 | Rejected |
| Grain protein | NMAXSO, NRESIDST | — | — | — | Incomplete | 10.1 | 1.94 | +2.45 | Rejected |
| Grain protein | NMAXSO, TCNT | — | — | — | Not started | 9.96 | 1.74 | +2.25 | Rejected |
| Phenology | TSUM1, TSUM2 | 839 | 99.68 | — | Major | 838 | 99.64 | — | Major |
| Phenology | TSUM1, TSUM2, DLO | 839 | 102.20 | +2.52 | Rejected | 838 | 102.16 | +2.52 | Rejected |
| Phenology | TSUM1, TSUM2, VERNBASE | 839 | 102.20 | +2.52 | Rejected | 838 | 102.16 | +2.52 | Rejected |
| Phenology | TSUM1, TSUM2, VERNSAT | 839 | 102.20 | +2.52 | Rejected | 838 | 102.16 | +2.52 | Rejected |
| Plant nitrogen | RNUPTAKEMAX | 0.398 | -25.49 | — | Major | 0.279 | -28.69 | — | Major |
| Plant nitrogen | RNUPTAKEMAX, NMAXSO | 0.379 | -22.51 | +2.99 | Rejected | 0.261 | -25.86 | +2.84 | Rejected |
| Yield | REALLOC_STEM_FRACTION | 3.56e+05 | 135.19 | — | Major | 2.03e+05 | 127.90 | — | Major |
| Yield | REALLOC_STEM_FRACTION, CVO | 3.4e+05 | 137.45 | +2.26 | Rejected | 1.91e+05 | 129.95 | +2.06 | Rejected |